<a href="https://colab.research.google.com/github/24f3005028/MLP-ASSIGNMENTS/blob/main/Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WEEK 3

In [1]:
!pip install -q gdown

import gdown
import pandas as pd
import numpy as np

file_id = "1REMgDd46i_klhGM2YZzim0lsc272twox"
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "dataset.csv", quiet=False)

df = pd.read_csv("dataset.csv")
print(df.shape)
print(df.head())
print(df.dtypes)

Downloading...
From: https://drive.google.com/uc?id=1REMgDd46i_klhGM2YZzim0lsc272twox
To: /content/dataset.csv
100%|██████████| 24.1k/24.1k [00:00<00:00, 36.6MB/s]

(748, 6)
    V1    V2       V3    V4        V5 Target
0  2.0  50.0  12500.0  98.0  NEGATIVE    YES
1  0.0  13.0   3250.0  28.0  NEGATIVE    YES
2    ?     ?   4000.0  35.0  NEGATIVE    YES
3    ?  20.0   5000.0  45.0  NEGATIVE    YES
4  1.0  24.0   6000.0  77.0  NEGATIVE     NO
V1         object
V2         object
V3        float64
V4        float64
V5         object
Target     object
dtype: object


In [3]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold, RFE, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
import numpy as np


target_col = "Target"

X = df.drop(columns=[target_col])
y_raw = df[[target_col]]

y_encoder = OrdinalEncoder()
y = y_encoder.fit_transform(y_raw).ravel()

X_numeric = X.iloc[:, :4].replace('?', np.nan).astype(float)
X_cat = X.iloc[:, 4:]

numeric_impute_idx = [0, 1]
numeric_scale_idx = [0, 1, 2, 3]
categorical_idx = [0]

imputer = SimpleImputer(strategy="mean")
X_numeric_imputed = X_numeric.copy()
X_numeric_imputed.iloc[:, numeric_impute_idx] = imputer.fit_transform(X_numeric.iloc[:, numeric_impute_idx])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric_imputed)

encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X_cat)

X_final = np.hstack([X_scaled, X_encoded])
print("Final feature matrix shape:", X_final.shape)
print("X_final preview (first row):", X_final[0])

Final feature matrix shape: (748, 5)
X_final preview (first row): [-0.93816939  7.70986653  7.62334626  2.61563344  0.        ]


In [4]:
# Q1
vt = VarianceThreshold(threshold=0.1)
X_vt = vt.fit_transform(X_final)
print("Variances per feature:", X_final.var(axis=0))
print("Features remaining after VarianceThreshold:", X_vt.shape[1])
print("Kept feature indices:", np.where(vt.get_support())[0])

Variances per feature: [1. 1. 1. 1. 0.]
Features remaining after VarianceThreshold: 4
Kept feature indices: [0 1 2 3]


In [5]:
#Q2
lr = LogisticRegression()
rfe = RFE(estimator=lr, n_features_to_select=2)
rfe.fit(X_final, y)
print("RFE selected feature indices:", np.where(rfe.support_)[0])
print("RFE ranking (lower = more important):", rfe.ranking_)

RFE selected feature indices: [0 2]
RFE ranking (lower = more important): [1 3 1 2 4]


In [6]:
#Q3
sfs_fwd = SequentialFeatureSelector(
    LogisticRegression(), n_features_to_select=2, direction="forward"
)
sfs_fwd.fit(X_final, y)
print("SFS forward selected feature indices:", np.where(sfs_fwd.get_support())[0])

SFS forward selected feature indices: [1 3]


In [7]:
#Q4
sfs_bwd = SequentialFeatureSelector(
    LogisticRegression(), n_features_to_select=2, direction="backward"
)
sfs_bwd.fit(X_final, y)
print("SFS backward selected feature indices:", np.where(sfs_bwd.get_support())[0])

SFS backward selected feature indices: [2 3]
